# B2-022-probabilistic-latent-models — Practice p24 — Solution

**Type:** challenge · **Difficulty:** advanced · **Concepts:** gaussian-reparameterization

*55 minutes.*  
**Set:** C  
**Program layer:** Round 2 extension  
**Compute:** `compute.policy: cpu` · seed `20260927`  
**Qualified prerequisites:** `book1:F1-scientific-python`, `book1:F3-matrices`, `book1:F4-multivar-calculus`, `book1:F5-probability`, `book1:F6-svd-spectral`, `book1:C1-ml-fundamentals`, `book1:C2-linear-models`, `book1:C5-neural-networks`, `book1:C6-pytorch`, `book1:C9-dimensionality-reduction`, `book1:C11-neural-training`, `B2-021-cross-modal-transformers-vision`  
**Remediation links actually used:** [book1:F1-scientific-python](../../../../book1/units/F1-scientific-python/lesson.ipynb), [book1:F3-matrices](../../../../book1/units/F3-matrices/lesson.ipynb), [book1:F4-multivar-calculus](../../../../book1/units/F4-multivar-calculus/lesson.ipynb), [book1:F5-probability](../../../../book1/units/F5-probability/lesson.ipynb), [book1:F6-svd-spectral](../../../../book1/units/F6-svd-spectral/lesson.ipynb), [book1:C1-ml-fundamentals](../../../../book1/units/C1-ml-fundamentals/lesson.ipynb), [book1:C2-linear-models](../../../../book1/units/C2-linear-models/lesson.ipynb), [book1:C5-neural-networks](../../../../book1/units/C5-neural-networks/lesson.ipynb), [book1:C6-pytorch](../../../../book1/units/C6-pytorch/lesson.ipynb), [book1:C9-dimensionality-reduction](../../../../book1/units/C9-dimensionality-reduction/lesson.ipynb), [book1:C11-neural-training](../../../../book1/units/C11-neural-training/lesson.ipynb), [B2-021-cross-modal-transformers-vision](../../B2-021-cross-modal-transformers-vision/lesson.ipynb).

## Supplied cell (copied verbatim from the statement)

In [ ]:
import importlib.util
from pathlib import Path

import torch
from torch import nn
import torch.nn.functional as F

SEED = 20260927
LOADER_CANDIDATES = (
    Path("../data/latent_data.py"),
    Path("units/B2-022-probabilistic-latent-models/data/latent_data.py"),
)
loader_path = next(path for path in LOADER_CANDIDATES if path.is_file())
loader_spec = importlib.util.spec_from_file_location("latent_data", loader_path)
latent_data = importlib.util.module_from_spec(loader_spec)
loader_spec.loader.exec_module(latent_data)
torch.set_num_threads(1)

import math

mu = torch.tensor([[0.5, -1.0]], requires_grad=True)
logvar = torch.tensor([[0.0, math.log(0.25)]], requires_grad=True)
eps = torch.tensor([[1.2, -0.4]])
ATOL = 1e-6
RTOL = 1e-6

## Solution

### (A) Leakage edges

| lines | what held-out information flows | into which training decision | class |
|---|---|---|---|
| 1–3 (consumed at 4, 9) | `mean`, `std` are computed over all 60 rows, including the 12 held-out rows | the normalization applied to every training input (and hence every gradient) | data leakage |
| 9 (with 5) | `heldout[:4]` — four held-out rows — are concatenated into every training batch | forward pass, `backward()`, and every `optimizer.step()` | data leakage |
| 16 | `heldout_negative_elbo(beta)` scores each candidate on held-out rows | the choice of $\beta$ (`best_beta`) | selection leakage |
| 17 (with 16) | the "final held-out score" is the minimum over candidates of the very score used to select them, on the same rows | the reported generalization estimate (optimistically biased; the held-out set is no longer untouched) | selection leakage |

(And because `heldout_negative_elbo` retrains with this pipeline, each candidate run also inherits the data leakage of lines 1–3 and 9.)

### (B) Non-differentiable sampling

**Line 12** draws `z` with `torch.distributions.Normal(...).sample()`, which returns a tensor detached from `mu` and `logvar`, so the reconstruction term on line 14 sends no gradient through `z` to the encoder.
The loss still decreases because (1) the KL part of `negative_elbo` is an explicit function of `mu` and `logvar` from line 11, so the encoder is trained — but only to push its posteriors toward $N(0,I)$; and (2) the decoder still receives the reconstruction gradient through `x_hat` and learns to map the (now uninformative) noise to roughly the data mean.
Both terms go down, but the encoder never learns to carry information, so the falling loss hides a collapsed latent.

### (B) Demonstration

In [ ]:
def blocked_sample(mu, logvar):
    return torch.distributions.Normal(mu, torch.exp(0.5 * logvar)).sample()   # reproduces line 12


def reparameterize(mu, logvar, eps):
    if not (mu.shape == logvar.shape == eps.shape):
        raise ValueError("mu, logvar, and eps must have identical shapes")
    return mu + torch.exp(0.5 * logvar) * eps


torch.manual_seed(SEED)
z_blocked = blocked_sample(mu, logvar)
z = reparameterize(mu, logvar, eps)
z.sum().backward()
print(z_blocked.requires_grad, mu.grad, logvar.grad)

### (C) Repaired pipeline

- **Splits.** Training IDs in stored order; the **validation subset is the last 8 training rows** (`TRAIN_IDS["mixture8d"][40:]`), and the remaining **40 rows** (`TRAIN_IDS["mixture8d"][:40]`) are the fit rows, so the repaired training batch has shape `(40, 8)`. Held-out rows are loaded only for the single final evaluation.
- **Normalization.** Mean and standard deviation from the 40 fit rows only, then applied unchanged to validation and held-out rows.
- **Forward.** `model(batch, eps)` with `eps` drawn from a `torch.Generator` seeded with `SEED`; no sampler inside the model.
- **$\beta$ selection.** Each candidate is trained from the same seed for 50 updates on the fit rows with loss $\overline{\text{recon}}+\beta\,\overline{\text{KL}}$; candidates are compared on the **validation** negative ELBO at $\beta=1$ (a common, $\beta$-independent scale), with fixed validation noise.
- **Final report.** The selected model is evaluated once on the held-out rows.

In [ ]:
DATASET = "mixture8d"
CANDIDATE_BETAS = (0.5, 1.0, 2.0)
N_UPDATES = 50
TRAIN_ID_TUPLE = latent_data.TRAIN_IDS[DATASET]
HELDOUT_ID_TUPLE = latent_data.HELDOUT_IDS[DATASET]
FIT_IDS = TRAIN_ID_TUPLE[:40]
VAL_IDS = TRAIN_ID_TUPLE[40:]


def _check_inputs(x, x_hat, mu, logvar):
    for name, t in (("x", x), ("x_hat", x_hat), ("mu", mu), ("logvar", logvar)):
        if t.dim() != 2:
            raise ValueError(f"{name} must be 2-D")
    if x.shape != x_hat.shape:
        raise ValueError("x and x_hat shapes differ")
    if mu.shape != logvar.shape:
        raise ValueError("mu and logvar shapes differ")
    if x.shape[0] != mu.shape[0]:
        raise ValueError("batch sizes differ")


def negative_elbo_terms(x, x_hat, mu, logvar):
    _check_inputs(x, x_hat, mu, logvar)
    recon = 0.5 * ((x - x_hat) ** 2).sum(dim=1)                       # sum over features
    kl = 0.5 * (mu ** 2 + logvar.exp() - logvar - 1).sum(dim=1)       # sum over latents
    return recon, kl


def negative_elbo(x, x_hat, mu, logvar):
    recon, kl = negative_elbo_terms(x, x_hat, mu, logvar)
    return (recon + kl).mean()                                         # mean over the batch

class TinyVAE(nn.Module):
    def __init__(self, in_dim=8, hidden=16, latent=2):
        super().__init__()
        self.hidden = nn.Linear(in_dim, hidden)
        self.mu_head = nn.Linear(hidden, latent)
        self.logvar_head = nn.Linear(hidden, latent)
        self.decoder = nn.Sequential(nn.Linear(latent, hidden), nn.ReLU(), nn.Linear(hidden, in_dim))

    def encode(self, x):
        h = F.relu(self.hidden(x))
        return self.mu_head(h), self.logvar_head(h)

    def decode(self, z):
        return self.decoder(z)

    def forward(self, x, eps):
        mu, logvar = self.encode(x)
        z = mu + torch.exp(0.5 * logvar) * eps
        return self.decode(z), mu, logvar


raw_fit = latent_data.load_rows(DATASET, FIT_IDS)
raw_val = latent_data.load_rows(DATASET, VAL_IDS)
fit_mean, fit_std = raw_fit.mean(dim=0), raw_fit.std(dim=0)


def normalize(rows):
    return (rows - fit_mean) / fit_std


fit_batch = normalize(raw_fit)
val_batch = normalize(raw_val)


def train_with_beta(beta):
    torch.manual_seed(SEED)
    model = TinyVAE(8, 16, 2)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.01, betas=(0.9, 0.999), eps=1e-8, weight_decay=0)
    eps_generator = torch.Generator().manual_seed(SEED)
    for _ in range(N_UPDATES):
        eps_batch = torch.randn(fit_batch.shape[0], 2, generator=eps_generator)
        optimizer.zero_grad(set_to_none=True)
        x_hat, mu_b, logvar_b = model(fit_batch, eps_batch)
        recon, kl = negative_elbo_terms(fit_batch, x_hat, mu_b, logvar_b)
        loss = recon.mean() + beta * kl.mean()
        loss.backward()
        optimizer.step()
    return model


def evaluate(model, batch, seed):
    eval_eps = torch.randn(batch.shape[0], 2, generator=torch.Generator().manual_seed(seed))
    with torch.no_grad():
        x_hat, mu_b, logvar_b = model(batch, eval_eps)
        return negative_elbo(batch, x_hat, mu_b, logvar_b).item()


models = {beta: train_with_beta(beta) for beta in CANDIDATE_BETAS}
validation_scores = {beta: evaluate(models[beta], val_batch, SEED + 1) for beta in CANDIDATE_BETAS}
best_beta = min(CANDIDATE_BETAS, key=validation_scores.get)

# Held-out rows are touched exactly once, here, after every decision is made.
raw_heldout = latent_data.load_rows(DATASET, HELDOUT_ID_TUPLE)
final_heldout_score = evaluate(models[best_beta], normalize(raw_heldout), SEED + 2)
print("validation scores", validation_scores, "best beta", best_beta)
print("final held-out negative ELBO", round(final_heldout_score, 4))

### (D) Hash check, before normalization

`latent_data.ROW_SHA256` stores hashes of the **raw** float32 rows. After normalization every row's bytes change, so a normalized row would match *neither* the training nor the held-out hashes — the assertion "no held-out row was used" would then pass vacuously, even for a pipeline that leaked. Hashing the raw rows the pipeline selects, before any transformation, is the only point where the hash identifies which stored row was used.

In [ ]:
hashes = latent_data.ROW_SHA256[DATASET]
train_hashes = {hashes[i] for i in TRAIN_ID_TUPLE}
heldout_hashes = {hashes[i] for i in HELDOUT_ID_TUPLE}
raw_selected = torch.cat([raw_fit, raw_val])
selected_hashes = [latent_data.row_sha256(row) for row in raw_selected]
normalized_hashes = [latent_data.row_sha256(row) for row in torch.cat([fit_batch, val_batch])]
print(len(selected_hashes), sum(h in train_hashes for h in normalized_hashes))

### Answer check

In [ ]:
# (B) probes
assert z_blocked.requires_grad is False
assert torch.allclose(mu.grad, torch.ones_like(mu), atol=ATOL, rtol=RTOL)
assert torch.allclose(logvar.grad, 0.5 * torch.exp(0.5 * logvar.detach()) * eps, atol=ATOL, rtol=RTOL)
# (D) hash check on raw rows
assert len(selected_hashes) == 48
assert all(h in train_hashes for h in selected_hashes)
assert not any(h in heldout_hashes for h in selected_hashes)
assert set(FIT_IDS).isdisjoint(VAL_IDS) and set(FIT_IDS + VAL_IDS) == set(TRAIN_ID_TUPLE)
assert not any(h in train_hashes or h in heldout_hashes for h in normalized_hashes)   # why hashing must precede normalization
# (C) repaired batch shape (train minus validation) and a finite final score
assert fit_batch.shape == (40, 8) and val_batch.shape == (8, 8)
assert best_beta in CANDIDATE_BETAS
assert all(math.isfinite(v) for v in validation_scores.values()) and math.isfinite(final_heldout_score)